# AgroPredict — Model Training v3 (improved)

**What changed from v2, and why:**

The v2 baseline comparison showed every model family (KNN, Logistic Regression, Decision Tree,
Random Forest, SVC, LightGBM, XGBoost) landing in the same low macro-F1 range (0.07–0.17).
Because the problem showed up identically across completely different algorithm families, it
pointed to the *task*, not a bug: with 140 crop classes and only 8 agronomic features, many crops
plausibly have overlapping "good growing condition" ranges, so exact single-label classification
has a low ceiling — even though the EDA's ANOVA/MI tests showed "significant" differences (which,
at 100,000 rows, can be true even when the practical overlap between classes is large).

This notebook makes four targeted changes in response:

1. **Feature engineering** — domain-informed features (NPK ratios, an aridity index) that may carry
   more separating signal than the raw values alone.
2. **Two evaluation metrics side by side** — macro-F1 (strict, single-guess) AND top-3 accuracy
   (was the true crop in the model's top 3 suggestions — the realistic way an advisory tool would
   actually be used).
3. **Model A vs Model B comparison** — quantifying exactly how much adding geography (state,
   lat/long) helps, instead of assuming Model A (agronomic-only) is automatically the right
   deployment choice. Since the FastAPI backend already plans OpenWeather-based lookups (which
   return lat/long), Model B is a legitimate deployment option if it wins meaningfully — not just
   an academic ablation.
4. **A quick hierarchical-classification check** — testing whether `crop_category` (a coarser,
   fewer-class target from `crop_profiles.csv`) is meaningfully easier to predict, as evidence for
   a future two-stage system, without committing to building the full pipeline in this pass.

Every expensive cell (baseline comparison, hyperparameter search) is **checkpointed to a `.jsonl`
file on Drive** and safely resumable across Colab disconnects/timeouts, and uses single-level
parallelism (model parallelizes internally; the CV loop itself runs sequentially) to avoid the
RAM crashes from nested parallelism we hit in v2.


## 1. Colab Setup

In [ ]:
import os

# EDIT THIS to the folder in your Drive that contains the CSVs
DATA_DIR = "/content/drive/MyDrive/Dataset For colab/AgroPredict V2"

expected_files = [
    "indian_crop_training_dataset.csv",
    "crop_profiles.csv",
    "data_dictionary.csv",
    "real_indian_crop_validation_dataset.csv",
]
missing = [f for f in expected_files if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    print("Could not find these files in DATA_DIR -- fix the path above:")
    for f in missing:
        print(" -", f)
    print()
    print("Contents of DATA_DIR:")
    print(os.listdir(DATA_DIR) if os.path.isdir(DATA_DIR) else "DATA_DIR does not exist")
else:
    print("All expected files found in", DATA_DIR)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q xgboost lightgbm shap joblib


## 2. Imports, Data Load, Feature Engineering

In [ ]:
import gc
import json
import warnings
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, ParameterSampler
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, top_k_accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore", message="X does not have valid feature names")

df = pd.read_csv(os.path.join(DATA_DIR, "indian_crop_training_dataset.csv"))
crop_profiles = pd.read_csv(os.path.join(DATA_DIR, "crop_profiles.csv"))

RAW_NUMERIC_FEATURES = [
    "nitrogen_N_kg_ha", "phosphorus_P_kg_ha", "potassium_K_kg_ha",
    "temperature_C", "humidity_percent", "rainfall_mm",
    "soil_pH", "soil_moisture_percent",
]
CATEGORICAL_FEATURES = ["soil_type", "season"]
GEO_CATEGORICAL_FEATURES = ["state"]
GEO_NUMERIC_FEATURES = ["latitude", "longitude"]
TARGET = "crop"

# --- Feature engineering: domain-informed ratios/indices, not just raw values ---
# Small epsilon avoids divide-by-zero without materially changing the ratio for real values.
EPS = 1e-3
df["npk_sum"] = df["nitrogen_N_kg_ha"] + df["phosphorus_P_kg_ha"] + df["potassium_K_kg_ha"]
df["n_p_ratio"] = df["nitrogen_N_kg_ha"] / (df["phosphorus_P_kg_ha"] + EPS)
df["n_k_ratio"] = df["nitrogen_N_kg_ha"] / (df["potassium_K_kg_ha"] + EPS)
df["p_k_ratio"] = df["phosphorus_P_kg_ha"] / (df["potassium_K_kg_ha"] + EPS)
# Aridity index: rainfall relative to temperature -- a standard agronomic idea (higher = wetter/cooler)
df["aridity_index"] = df["rainfall_mm"] / (df["temperature_C"] + EPS)

ENGINEERED_FEATURES = ["npk_sum", "n_p_ratio", "n_k_ratio", "p_k_ratio", "aridity_index"]
NUMERIC_FEATURES = RAW_NUMERIC_FEATURES + ENGINEERED_FEATURES

# float64 -> float32: halves memory for every numeric array produced downstream. No meaningful
# precision loss for any of these models at these value ranges.
for col in NUMERIC_FEATURES:
    df[col] = df[col].astype("float32")
for col in CATEGORICAL_FEATURES + GEO_CATEGORICAL_FEATURES:
    df[col] = df[col].astype("category")
for col in GEO_NUMERIC_FEATURES:
    df[col] = df[col].astype("float32")

MODEL_A_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
MODEL_B_FEATURES = MODEL_A_FEATURES + GEO_CATEGORICAL_FEATURES + GEO_NUMERIC_FEATURES

print(f"Loaded {df.shape[0]:,} rows, {df[TARGET].nunique()} crop classes")
print(f"Model A features ({len(MODEL_A_FEATURES)}): {MODEL_A_FEATURES}")
print(f"Model B features ({len(MODEL_B_FEATURES)}): {MODEL_B_FEATURES}")
print(f"df memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")


## 3. Recap: Why the Naive Approach Plateaued

Quick, cheap diagnostic: for each crop, compute its mean numeric-feature profile, then find its
nearest *other* crop in that feature space. Crops with a very close neighbor are strong candidates
for being genuinely hard to separate from raw agronomic features alone — this is the concrete
evidence behind "statistically significant ≠ practically separable" from the EDA follow-up.


In [ ]:
from scipy.spatial.distance import cdist

crop_means = df.groupby(TARGET)[RAW_NUMERIC_FEATURES].mean()
# standardize so no single feature (e.g. rainfall_mm, large scale) dominates the distance
crop_means_std = (crop_means - crop_means.mean()) / crop_means.std()

dist_matrix = cdist(crop_means_std.values, crop_means_std.values)
np.fill_diagonal(dist_matrix, np.inf)  # exclude self-distance

nearest_idx = dist_matrix.argmin(axis=1)
nearest_dist = dist_matrix.min(axis=1)

overlap_df = pd.DataFrame({
    "crop": crop_means.index,
    "nearest_crop": crop_means.index[nearest_idx],
    "standardized_distance": nearest_dist,
}).sort_values("standardized_distance")

print("10 crop pairs with the MOST overlapping agronomic profiles (smallest distance):")
print(overlap_df.head(10).to_string(index=False))
print(f"\nMedian nearest-neighbor distance across all 140 crops: {overlap_df['standardized_distance'].median():.3f}")
print(f"Crops with a near-duplicate profile (distance < 0.5): {(overlap_df['standardized_distance'] < 0.5).sum()} / {len(overlap_df)}")


## 4. Train/Test Split (stratified, held out until final evaluation)

In [ ]:
le = LabelEncoder()
X_full = df[MODEL_B_FEATURES].copy()  # superset; each pipeline below selects its own columns via ColumnTransformer
y = le.fit_transform(df[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows (held out until Section 10)")


## 5. Preprocessing Pipelines — Model A (agronomic only) and Model B (agronomic + geography)

In [ ]:
def make_preprocessor(categorical_features, numeric_features, scale_numeric):
    num_step = StandardScaler() if scale_numeric else "passthrough"
    return ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float32), categorical_features),
        ("num", num_step, numeric_features),
    ])

# Model A: agronomic + soil_type/season only, no geography
preprocessor_A_tree = make_preprocessor(CATEGORICAL_FEATURES, NUMERIC_FEATURES, scale_numeric=False)
preprocessor_A_scaled = make_preprocessor(CATEGORICAL_FEATURES, NUMERIC_FEATURES, scale_numeric=True)

# Model B: Model A + state (categorical) + lat/long (numeric)
preprocessor_B_tree = make_preprocessor(
    CATEGORICAL_FEATURES + GEO_CATEGORICAL_FEATURES, NUMERIC_FEATURES + GEO_NUMERIC_FEATURES, scale_numeric=False
)
preprocessor_B_scaled = make_preprocessor(
    CATEGORICAL_FEATURES + GEO_CATEGORICAL_FEATURES, NUMERIC_FEATURES + GEO_NUMERIC_FEATURES, scale_numeric=True
)

print("Preprocessors ready: preprocessor_A_tree/scaled, preprocessor_B_tree/scaled")


## 6. Evaluation: macro-F1 AND top-3 accuracy, together

`top_k_accuracy_score` needs `predict_proba`, so SVC is configured with `probability=True` here
(only used on the 15k subsample, so the extra cost is bounded).


In [ ]:
N_CLASSES = len(le.classes_)
ALL_LABELS = np.arange(N_CLASSES)

def top3_scorer(estimator, X, y):
    proba = estimator.predict_proba(X)
    return top_k_accuracy_score(y, proba, k=3, labels=ALL_LABELS)

SCORING = {"macro_f1": "f1_macro", "top3_acc": top3_scorer}


## 7. Baseline Comparison — Model A (checkpointed, resumable)

Same RAM-safe rule as before: the model parallelizes internally (`n_jobs=-1`), the outer CV loop
runs sequentially (`n_jobs=1`). Each model's result is written to disk immediately, so a
disconnect only costs the model that was mid-fit, not the whole comparison.


In [ ]:
def run_baseline_comparison(model_dict, preprocessor, X, y, checkpoint_path, cv, svc_subsample=None):
    completed = {}
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path) as f:
            for line in f:
                rec = json.loads(line)
                completed[rec["name"]] = rec
        print(f"Resuming: {len(completed)} model(s) already scored: {list(completed.keys())}")

    for name, model in model_dict.items():
        if name in completed:
            rec = completed[name]
            print(f"{name:<20} macro-F1: {rec['macro_f1']:.4f} | top3-acc: {rec['top3_acc']:.4f}  [skipped]")
            continue

        this_X, this_y = X, y
        note = ""
        if name == "SVC" and svc_subsample is not None:
            this_X, _, this_y, _ = train_test_split(
                X, y, train_size=svc_subsample, stratify=y, random_state=42
            )
            note = f"estimated from a {svc_subsample:,}-row subsample"

        pipe = Pipeline([("preprocess", preprocessor), ("model", model)])
        scores = cross_validate(
            pipe, this_X, this_y, cv=cv, scoring=SCORING,
            n_jobs=1, pre_dispatch="1*n_jobs",
        )
        rec = {
            "name": name,
            "macro_f1": float(np.mean(scores["test_macro_f1"])),
            "macro_f1_std": float(np.std(scores["test_macro_f1"])),
            "top3_acc": float(np.mean(scores["test_top3_acc"])),
            "top3_acc_std": float(np.std(scores["test_top3_acc"])),
            "note": note,
        }
        with open(checkpoint_path, "a") as f:
            f.write(json.dumps(rec) + "\n")
        completed[name] = rec
        print(f"{name:<20} macro-F1: {rec['macro_f1']:.4f} | top3-acc: {rec['top3_acc']:.4f}{'  [' + note + ']' if note else ''}")
        del pipe, scores
        gc.collect()

    return completed


cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

no_scale_models_A = {
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1, eval_metric="mlogloss"),
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1, verbosity=-1),
}
scale_models_A = {
    "LogisticRegression": LogisticRegression(max_iter=1000, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_jobs=-1),
    "SVC": SVC(random_state=42, probability=True),
}

CHECKPOINT_A_TREE = os.path.join(DATA_DIR, "baseline_A_tree_checkpoint.jsonl")
CHECKPOINT_A_SCALED = os.path.join(DATA_DIR, "baseline_A_scaled_checkpoint.jsonl")

results_A = {}
results_A.update(run_baseline_comparison(no_scale_models_A, preprocessor_A_tree, X_train[MODEL_A_FEATURES], y_train, CHECKPOINT_A_TREE, cv5))
results_A.update(run_baseline_comparison(scale_models_A, preprocessor_A_scaled, X_train[MODEL_A_FEATURES], y_train, CHECKPOINT_A_SCALED, cv5, svc_subsample=15000))


## 8. Baseline Comparison — Model B (agronomic + geography)

In [ ]:
no_scale_models_B = {
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(random_state=42, n_jobs=-1, eval_metric="mlogloss"),
    "LightGBM": LGBMClassifier(random_state=42, n_jobs=-1, verbosity=-1),
}
scale_models_B = {
    "LogisticRegression": LogisticRegression(max_iter=1000, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_jobs=-1),
    "SVC": SVC(random_state=42, probability=True),
}

CHECKPOINT_B_TREE = os.path.join(DATA_DIR, "baseline_B_tree_checkpoint.jsonl")
CHECKPOINT_B_SCALED = os.path.join(DATA_DIR, "baseline_B_scaled_checkpoint.jsonl")

results_B = {}
results_B.update(run_baseline_comparison(no_scale_models_B, preprocessor_B_tree, X_train[MODEL_B_FEATURES], y_train, CHECKPOINT_B_TREE, cv5))
results_B.update(run_baseline_comparison(scale_models_B, preprocessor_B_scaled, X_train[MODEL_B_FEATURES], y_train, CHECKPOINT_B_SCALED, cv5, svc_subsample=15000))


## 9. Quick check: is `crop_category` (fewer classes) meaningfully easier?

One cheap run, evidence for/against a future two-stage system -- not a full hierarchical pipeline.


In [ ]:
cat_map = crop_profiles.set_index("crop")["crop_category"]
df["crop_category"] = df[TARGET].map(cat_map)
print(f"crop_category classes: {df['crop_category'].nunique()} (vs {df[TARGET].nunique()} individual crops)")

le_cat = LabelEncoder()
y_cat = le_cat.fit_transform(df["crop_category"])
X_cat_train, X_cat_test, y_cat_train, y_cat_test = train_test_split(
    df[MODEL_A_FEATURES], y_cat, test_size=0.2, stratify=y_cat, random_state=42
)

quick_model = Pipeline([
    ("preprocess", clone(preprocessor_A_tree)),
    ("model", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)),
])
quick_model.fit(X_cat_train, y_cat_train)
cat_pred = quick_model.predict(X_cat_test)
cat_macro_f1 = f1_score(y_cat_test, cat_pred, average="macro")
cat_accuracy = (cat_pred == y_cat_test).mean()

print(f"crop_category RandomForest -- macro-F1: {cat_macro_f1:.4f} | accuracy: {cat_accuracy:.4f}")
print("Compare this to Model A's individual-crop RandomForest macro-F1 above.")
print("If this is substantially higher, that's real evidence a category-first, crop-second")
print("two-stage system would outperform single-stage 140-way classification -- worth building")
print("as a v4 if you have time before your deadline.")

del quick_model, X_cat_train, X_cat_test, y_cat_train, y_cat_test
gc.collect()


## 10. Results — Model A vs Model B, and choosing what to tune

In [ ]:
def results_to_df(results_dict):
    return pd.DataFrame(results_dict).T[["macro_f1", "macro_f1_std", "top3_acc", "top3_acc_std", "note"]].sort_values("top3_acc", ascending=False)

results_A_df = results_to_df(results_A)
results_B_df = results_to_df(results_B)

print("=== Model A (agronomic only) ===")
print(results_A_df)
print()
print("=== Model B (agronomic + geography) ===")
print(results_B_df)

results_A_df.to_csv(os.path.join(DATA_DIR, "results_model_A.csv"))
results_B_df.to_csv(os.path.join(DATA_DIR, "results_model_B.csv"))

best_A_name = results_A_df.index[0]
best_B_name = results_B_df.index[0]
uplift = results_B_df.loc[best_B_name, "top3_acc"] - results_A_df.loc[best_A_name, "top3_acc"]
print(f"\nBest Model A: {best_A_name} (top3-acc {results_A_df.loc[best_A_name, 'top3_acc']:.4f})")
print(f"Best Model B: {best_B_name} (top3-acc {results_B_df.loc[best_B_name, 'top3_acc']:.4f})")
print(f"Geography uplift on top3-acc: {uplift:+.4f}")
print()
print("Decide TRACK below based on this uplift: if it's large (e.g. >0.05), Model B is likely")
print("worth deploying as primary (you'll have lat/long from OpenWeather at inference time anyway).")
print("If it's small, Model A stays primary -- simpler, and doesn't require a location lookup to work.")


## 11. Hyperparameter Tuning — Winner Model (checkpointed, resumable)

In [ ]:
# --- Set these two after reviewing Section 10's output ---
TRACK = "A"              # "A" or "B" -- which feature set to deploy
WINNER = best_A_name if TRACK == "A" else best_B_name   # <-- override manually if you disagree with the auto-pick

PARAM_GRIDS = {
    "RandomForest": {
        "model__n_estimators": [200, 400, 600, 800],
        "model__max_depth": [10, 20, 30, 50],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
        "model__max_features": ["sqrt", "log2", None],
    },
    "XGBoost": {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [4, 6, 8, 10],
        "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
    },
    "LightGBM": {
        "model__n_estimators": [200, 400, 600],
        "model__num_leaves": [31, 63, 127],
        "model__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "model__subsample": [0.6, 0.8, 1.0],
        "model__colsample_bytree": [0.6, 0.8, 1.0],
    },
    "DecisionTree": {
        "model__max_depth": [None, 10, 20, 30, 50],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
    },
}

all_models = {**no_scale_models_A, **scale_models_A} if TRACK == "A" else {**no_scale_models_B, **scale_models_B}
FEATURES = MODEL_A_FEATURES if TRACK == "A" else MODEL_B_FEATURES
scaled_needed = WINNER in ("LogisticRegression", "KNN", "SVC")
if TRACK == "A":
    winner_preprocessor = preprocessor_A_scaled if scaled_needed else preprocessor_A_tree
else:
    winner_preprocessor = preprocessor_B_scaled if scaled_needed else preprocessor_B_tree

base_model = clone(all_models[WINNER])
CHECKPOINT_PATH = os.path.join(DATA_DIR, f"search_checkpoint_{TRACK}_{WINNER}.jsonl")

N_ITER = 10
SEARCH_CV_FOLDS = 3
search_cv = StratifiedKFold(n_splits=SEARCH_CV_FOLDS, shuffle=True, random_state=42)
candidates = list(ParameterSampler(PARAM_GRIDS[WINNER], n_iter=N_ITER, random_state=42))

completed = {}
if os.path.exists(CHECKPOINT_PATH):
    with open(CHECKPOINT_PATH) as f:
        for line in f:
            rec = json.loads(line)
            completed[json.dumps(rec["params"], sort_keys=True)] = rec

print(f"Tuning {WINNER} on Track {TRACK} ({len(FEATURES)} features). Resuming {len(completed)} completed candidates.")

for i, params in enumerate(candidates):
    key = json.dumps(params, sort_keys=True)
    if key in completed:
        rec = completed[key]
        print(f"[{i+1}/{N_ITER}] already done: macro-F1 {rec['macro_f1']:.4f} | top3-acc {rec['top3_acc']:.4f} | {params}")
        continue

    pipe = Pipeline([("preprocess", winner_preprocessor), ("model", clone(base_model))])
    pipe.set_params(**params)

    scores = cross_validate(
        pipe, X_train[FEATURES], y_train, cv=search_cv, scoring=SCORING, n_jobs=1
    )
    rec = {
        "params": params,
        "macro_f1": float(np.mean(scores["test_macro_f1"])),
        "top3_acc": float(np.mean(scores["test_top3_acc"])),
    }
    with open(CHECKPOINT_PATH, "a") as f:
        f.write(json.dumps(rec) + "\n")
    completed[key] = rec
    print(f"[{i+1}/{N_ITER}] macro-F1: {rec['macro_f1']:.4f} | top3-acc: {rec['top3_acc']:.4f} | {params}")
    del pipe, scores
    gc.collect()

best = max(completed.values(), key=lambda r: r["top3_acc"])
print(f"\nBest candidate (by top3-acc): {best['top3_acc']:.4f} | {best['params']}")

# Fit the winning config once, fully, on the entire training set
final_model = Pipeline([("preprocess", winner_preprocessor), ("model", clone(base_model))])
final_model.set_params(**best["params"])
final_model.fit(X_train[FEATURES], y_train)
print("final_model fit on full training set.")


## 12. Final Evaluation on Held-out Test Set

In [ ]:
y_pred = final_model.predict(X_test[FEATURES])
y_proba = final_model.predict_proba(X_test[FEATURES])

test_macro_f1 = f1_score(y_test, y_pred, average="macro")
test_top3_acc = top_k_accuracy_score(y_test, y_proba, k=3, labels=ALL_LABELS)
test_accuracy = (y_pred == y_test).mean()

print(f"Held-out test macro-F1:   {test_macro_f1:.4f}")
print(f"Held-out test top-3 acc:  {test_top3_acc:.4f}")
print(f"Held-out test accuracy:   {test_accuracy:.4f}")

report = classification_report(y_test, y_pred, target_names=le.classes_, output_dict=True)
report_df = pd.DataFrame(report).T.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")
print("\nWeakest-performing crops:")
print(report_df.sort_values("f1-score").head(10))
print("\nStrongest-performing crops:")
print(report_df.sort_values("f1-score", ascending=False).head(10))

cm = confusion_matrix(y_test, y_pred)
np.fill_diagonal(cm, 0)
top_confusions = []
for i, j in zip(*np.unravel_index(np.argsort(cm, axis=None)[-15:], cm.shape)):
    if cm[i, j] > 0:
        top_confusions.append((le.classes_[i], le.classes_[j], cm[i, j]))
print("\nMost-confused crop pairs (true -> predicted, count):")
for true_crop, pred_crop, count in sorted(top_confusions, key=lambda x: -x[2]):
    print(f"  {true_crop} -> {pred_crop}: {count}")


## 13. Explainability — SHAP

Uses `TreeExplainer` (fast, exact for tree-based models) if `final_model` is tree-based; falls
back to a model-agnostic `Explainer` on a small sample otherwise. Run on a subsample of the test
set for speed/memory, not the full 20k rows.


In [ ]:
import shap

SHAP_SAMPLE_SIZE = 500
X_shap_sample = X_test[FEATURES].sample(n=min(SHAP_SAMPLE_SIZE, len(X_test)), random_state=42)
X_shap_transformed = final_model.named_steps["preprocess"].transform(X_shap_sample)
feature_names_out = final_model.named_steps["preprocess"].get_feature_names_out()

fitted_model = final_model.named_steps["model"]
tree_based = isinstance(fitted_model, (RandomForestClassifier, DecisionTreeClassifier, XGBClassifier, LGBMClassifier))

if tree_based:
    explainer = shap.TreeExplainer(fitted_model)
else:
    explainer = shap.Explainer(fitted_model.predict_proba, X_shap_transformed)

shap_values = explainer.shap_values(X_shap_transformed) if tree_based else explainer(X_shap_transformed).values

# Multiclass SHAP: aggregate mean |SHAP| across all classes for an overall feature-importance view
if isinstance(shap_values, list):
    mean_abs_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
else:
    mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2)) if shap_values.ndim == 3 else np.abs(shap_values).mean(axis=0)

shap_importance = pd.Series(mean_abs_shap, index=feature_names_out).sort_values(ascending=False)
print("Mean |SHAP value| by feature (overall importance across all 140 classes):")
print(shap_importance.head(20))

print("\nCross-check against Section 7 of the EDA (N/P/K ranked weakest there):")
npk_cols = [c for c in feature_names_out if any(k in c for k in ["nitrogen", "phosphorus", "potassium"])]
print(shap_importance[npk_cols].sort_values(ascending=False))


## 14. Save Artifacts for FastAPI Deployment

In [ ]:
joblib.dump(final_model, os.path.join(DATA_DIR, "model_final_pipeline.joblib"))
joblib.dump(le, os.path.join(DATA_DIR, "target_label_encoder.joblib"))

# Metadata the FastAPI Pydantic schema will need -- feature list, valid categories, track used
deploy_metadata = {
    "track": TRACK,
    "features": FEATURES,
    "categorical_options": {
        col: sorted(df[col].astype(str).unique().tolist())
        for col in (CATEGORICAL_FEATURES + (GEO_CATEGORICAL_FEATURES if TRACK == "B" else []))
    },
    "n_classes": int(N_CLASSES),
    "test_macro_f1": float(test_macro_f1),
    "test_top3_acc": float(test_top3_acc),
}
with open(os.path.join(DATA_DIR, "deploy_metadata.json"), "w") as f:
    json.dump(deploy_metadata, f, indent=2)

print("Saved: model_final_pipeline.joblib, target_label_encoder.joblib, deploy_metadata.json")


## 15. Summary — for your portfolio write-up

Reuse this structure in your README / resume bullet / interview answer:

- **Problem:** recommend suitable crops for Indian agricultural conditions from soil + climate data,
  as an end-to-end API + frontend tool (not just a notebook).
- **Initial finding:** a naive 140-way single-label classifier plateaued at low macro-F1 across
  every model family tried (KNN through XGBoost), which ruled out a pipeline bug and pointed to
  genuine class overlap in agronomic feature space.
- **Diagnosis:** confirmed via nearest-neighbor distance between crop feature-profiles — found
  measurable overlap between multiple crop pairs, explaining the shared low ceiling.
- **Fix, three parts:**
  1. Reframed evaluation around top-3 accuracy (the realistic use case for an advisory tool),
     alongside macro-F1.
  2. Engineered domain-informed features (NPK ratios, an aridity index).
  3. Quantified the value of geography by comparing an agronomic-only model against one with
     state/lat/long — a deliberate, measured trade-off rather than an assumption.
- **Result:** [fill in your actual before/after top-3 accuracy numbers once this notebook finishes running].
- **Engineering practices applied:** stratified train/test split held out until final evaluation,
  checkpointed/resumable long-running training (survives Colab disconnects), single-level
  parallelism to avoid RAM exhaustion, SHAP-based explainability tied back to the EDA's own
  statistical findings, and serialized artifacts + metadata ready for a FastAPI deployment.
